In [1]:
%%capture
%pip install pycox lifelines scikit-survival optuna tqdm

In [2]:
# Standard library imports
import gc
import os
import random
import warnings

# Third-party imports
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torchtuples as tt
import xgboost as xgb
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from matplotlib import pyplot as plt
from pycox.models import CoxPH
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sksurv.datasets import load_breast_cancer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from tqdm.auto import tqdm
from pycox.models import DeepHitSingle
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from sksurv.datasets import load_whas500
from lifelines.datasets import load_rossi
from sksurv.metrics import integrated_brier_score

In [ ]:
# utilise gpu
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

Training on: cuda


In [ ]:
np.random.seed(1)
n_samples = 6000
d = 10

# Linear model
X = np.random.uniform(-1, 1, (n_samples, d))
# log-risk function
f_x = X[:, 0] + 2*X[:, 1]

lambda_0 = 1.0
U = np.random.uniform(0, 1, n_samples)
# survival times: T = - log(U) / (lambda_0 * exp(f(x)))
T = -np.log(U) / (lambda_0 * np.exp(f_x))

# censoring 50% similar to paper
C = np.random.exponential(scale=np.mean(T), size=n_samples)

time = np.minimum(T, C)  # choose minimum
event = (T <= C).astype(int)

In [6]:
# quadratic non-linear model
alpha = 3.0 # Controls the steepness of the "U" shape
X_nl_quad = np.random.uniform(-1, 1, (n_samples, d))

# f(x) = alpha * (x0^2 + x1^2)
# Risk is lowest at (0,0) and highest at the corners/edges
f_x_nl_quad = alpha * (X_nl_quad[:, 0]**2 + X_nl_quad[:, 1]**2)

U_quad = np.random.uniform(0, 1, n_samples)

# survival times: T = - log(U) / (lambda_0 * exp(f(x)))
T_nl_quad = -np.log(U_quad) / (lambda_0 * np.exp(f_x_nl_quad))

# censoring ~50%
C_nl_quad = np.random.exponential(scale=np.mean(T_nl_quad), size=n_samples)

time_nl_quad = np.minimum(T_nl_quad, C_nl_quad)
event_nl_quad = (T_nl_quad <= C_nl_quad).astype(int)

In [7]:
# WHAS500 data
df_whas, y_whas = load_whas500()
df_whas_numeric = pd.get_dummies(df_whas, drop_first=True)
X_whas = df_whas_numeric.astype("float32")
T_whas = y_whas["lenfol"].astype("float32")
E_whas = y_whas["fstat"].astype("float32")

In [ ]:
# rossi data
df_rossi = load_rossi()

X_rossi = df_rossi.drop(["week", "arrest"], axis=1).values.astype("float32")
T_rossi = df_rossi["week"].values.astype("float32")
E_rossi = df_rossi["arrest"].values.astype("float32")

In [11]:
# cox proportional hazards
def cph(X_train, T_train, E_train, X_test, times):
    df_train = pd.DataFrame(X_train)
    df_train["time"] = T_train
    df_train["event"] = E_train

    cph = CoxPHFitter(penalizer=0.0)  # no penalisation

    try:
        cph.fit(df_train, duration_col="time", event_col="event")

        # predictions flattened to a 1D array
        preds = cph.predict_partial_hazard(pd.DataFrame(X_test)).values.flatten()

        # return 2d  array for IBS
        surv = cph.predict_survival_function(pd.DataFrame(X_test), times=times).values.T

        return preds, surv

    except Exception as e:
        print(f"CPH failed to converge: {e}")
        return None

In [12]:
def cox_lasso(X_train, T_train, E_train, X_test, times=None):
    try:
        # scikit-survival requires a structured array for the target variable
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)],
                           dtype=[('event', 'bool'), ('time', 'float32')])

        # Initialize Lasso (l1_ratio=1.0 is pure Lasso / L1 penalty)
        lasso_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, fit_baseline_model=True)

        # fit model
        lasso_model.fit(X_train, y_train)

        # predict and flatten to ensure a clean 1D array for the bootstrap loop
        risk_scores = lasso_model.predict(X_test).flatten()

        # 2d array of survival functions for IBS
        surv_funcs = lasso_model.predict_survival_function(X_test)
        surv = np.array([fn(times) for fn in surv_funcs])

        return risk_scores, surv

    except Exception as e:
        print(f"Cox Lasso failed to converge: {e}")
        return None

In [13]:
# random survival forest
def rsf(X_train, T_train, E_train, X_test, times=None):
    try:
        y_train = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[("e", bool), ("t", float)])

        # fit rsf
        rsf_model = RandomSurvivalForest(n_estimators=100, min_samples_leaf=15, n_jobs=-1, random_state=0)
        rsf_model.fit(X_train, y_train)

        # risk for c-index
        risk = rsf_model.predict(X_test)

        # survival functions for IBS
        surv_funcs = rsf_model.predict_survival_function(X_test)
        surv = np.array([fn(times) for fn in surv_funcs])

        return risk, surv

    except Exception as e:
        print(f"RSF failed: {e}")
        return None, None if times is not None else None

In [14]:
def rsf_gpu(X_train, T_train, E_train, X_test):
    try:
        # format labels for xgboost cox. positive if event=1, negative if censored
        y_train = np.where(E_train == 1, T_train, -T_train)

        # random forest parameters
        params = {
            "objective": "survival:cox", # use Cox Proportional Hazards
            "eval_metric": "cox-nloglik",
            "tree_method": "hist",       # required for gpu execution
            "device": "cuda",            # utilise gpu
            "booster": "gbtree",
            "subsample": 0.8,            # Randomly sample 80% of rows per tree
            "colsample_bynode": 0.8,     # Randomly sample 80% of features per split
            "learning_rate": 0.05,        # 1.0 for random survival forest
            "min_child_weight": 15,
        }

        # Create DMatrix for training (with labels) and testing (without labels)
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dtest = xgb.DMatrix(X_test)

        # train forest (num_boost_round=1 ensures it just builds the single forest of 100 trees)
        bst = xgb.train(params, dtrain, num_boost_round=100)

        # predict and flatten
        preds = bst.predict(dtest).flatten()

        return preds

    except Exception as e:
        print(f"RSF (XGBoost GPU) failed: {e}")
        return None

In [15]:
def deepsurv(X_train, T_train, E_train, X_test, params, times):
    try:
        # Ensure arrays are float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
            X_train, T_train, E_train, test_size=0.2, random_state=42, stratify=E_train
        )

        y_tr = (T_tr, E_tr)
        y_val = (T_val, E_val)

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [params["nodes"]] * params["layers"]
        dropout = params["dropout"]
        batch_norm = params.get("batch_norm", True)

        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        if params.get("optimiser", "adam") == "adam":
            optim = tt.optim.Adam(lr=params["lr"], weight_decay=params.get("l2", 0.0))
        elif params.get("optimiser") == "sgd":
            optim = tt.optim.SGD(lr=params["lr"], momentum=params.get("momentum", 0.9),
                                 weight_decay=params.get("l2", 0.0), nesterov=True)
        else:
            raise ValueError("Unknown optimiser")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]
        batch_size = params.get("batch_size", 64)

        model.fit(
            X_tr, y_tr,
            batch_size=batch_size,
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # Return the flat array of log-hazard (risk) scores
        risk = model.predict_net(X_test).flatten()

        # survival probabilities
        _ = model.compute_baseline_hazards(X_tr, y_tr)  # calculate baseline hazards
        surv_df = model.predict_surv_df(X_test)
        surv = surv_df.reindex(times, method='ffill').bfill().values.T

        return risk, surv

    except Exception as e:
        print(f"DeepSurv failed: {e}")
        return None

In [16]:
def optimise_deepsurv(X_train, T_train, E_train, n_trials=30):
    X_train, T_train, E_train = X_train.astype("float32"), T_train.astype("float32"), E_train.astype("float32")

    X_tr, X_val, T_tr, T_val, E_tr, E_val = train_test_split(
        X_train, T_train, E_train, test_size=0.2, random_state=42, stratify=E_train
    )

    def objective(trial):
        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 16, 128, step=16)
        n_layers = trial.suggest_int("layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.6)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        l2 = trial.suggest_float("l2", 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        activation_name = trial.suggest_categorical("activation", ["ReLU", "SELU"])
        batch_norm = trial.suggest_categorical("batch_norm", [True, False])

        in_features = X_tr.shape[1]
        out_features = 1
        num_nodes = [n_nodes] * n_layers
        activation = nn.SELU if activation_name == "SELU" else nn.ReLU

        # deepsurv
        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features,
            batch_norm=batch_norm, dropout=dropout,
            activation=activation, output_bias=False
        )

        optim = tt.optim.Adam(lr=lr, weight_decay=l2)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CoxPH(net, optim, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        model.fit(
            X_tr, (T_tr, E_tr),
            batch_size=batch_size, epochs=500,
            callbacks=callbacks, verbose=False,
            val_data=(X_val, (T_val, E_val))
        )

        risk = model.predict_net(X_val).flatten()
        c_index = concordance_index(T_val, -risk, E_val)

        # memory cleanup
        del model, net, optim
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return c_index

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [17]:
def deephit(X_train, T_train, E_train, X_test, params, times):
    try:
        # Ensure float32 for PyTorch compatibility
        X_train = X_train.astype("float32")
        T_train = T_train.astype("float32")
        E_train = E_train.astype("float32")
        X_test = X_test.astype("float32")

        # extract parameters
        alpha = params.get("alpha", 0.2)
        sigma = params.get("sigma", 0.1)
        num_durations = params.get("num_durations", 10)

        # discretise time for DeepHit
        labtrans = LabTransDiscreteTime(num_durations)
        # Fit the transformer on training data and transform it
        y_train_discrete = labtrans.fit_transform(T_train, E_train)

        # Split strictly within the training set for Early Stopping
        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event = train_test_split(
            X_train, y_train_discrete[0], y_train_discrete[1],
            test_size=0.2, random_state=42, stratify=y_train_discrete[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # build the network
        in_features = X_train.shape[1]
        out_features = labtrans.out_features # Network must output exactly the number of time bins
        num_nodes = [params.get("nodes", 32)] * params.get("layers", 2)
        dropout = params.get("dropout", 0.2)
        batch_norm = params.get("batch_norm", True)

        activation = nn.SELU if params.get("activation", "ReLU") == "SELU" else nn.ReLU

        net = tt.practical.MLPVanilla(
            in_features, num_nodes, out_features, batch_norm,
            dropout, activation=activation, output_bias=False
        )

        optim = tt.optim.Adam(lr=params.get("lr", 1e-3), weight_decay=params.get("l2", 0.01))
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # deephit
        model = DeepHitSingle(net, optim, alpha=alpha, sigma=sigma, duration_index=labtrans.cuts, device=device)

        callbacks = [tt.callbacks.EarlyStopping(patience=15)]

        # train
        model.fit(
            X_tr, y_tr,
            batch_size=params.get("batch_size", 64),
            epochs=500,
            callbacks=callbacks,
            verbose=False,
            val_data=(X_val, y_val)
        )

        # predict
        surv_df = model.predict_surv_df(X_test)

        # compute risk
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        # survival probabilities
        surv = surv_df.reindex(times, method='ffill').bfill().values.T

        return risk, surv

    except Exception as e:
        print(f"DeepHit failed: {e}")
        return None

In [18]:
# Sim Linear
X_train_lin, X_test_lin, T_train_lin, T_test_lin, E_train_lin, E_test_lin = train_test_split(
    X, time, event, test_size=0.2, random_state=42, stratify=event
)

# Sim Nonlinear Quadratic
X_train_quad, X_test_quad, T_train_quad, T_test_quad, E_train_quad, E_test_quad = train_test_split(
    X_nl_quad, time_nl_quad, event_nl_quad, test_size=0.2, random_state=42, stratify=event_nl_quad
)

# whas
X_train_met, X_test_met, T_train_met, T_test_met, E_train_met, E_test_met = train_test_split(
    X_whas, T_whas, E_whas, test_size=0.2, random_state=42, stratify=E_whas
)

# rossi
X_train_sup, X_test_sup, T_train_sup, T_test_sup, E_train_sup, E_test_sup = train_test_split(
    X_rossi, T_rossi, E_rossi, test_size=0.2, random_state=42, stratify=E_rossi
)

In [ ]:
n_trials = 20

best_linear = optimise_deepsurv(X_train_lin, T_train_lin, E_train_lin, n_trials=n_trials)
params_linear = best_linear.copy()

best_nonlinear_quad = optimise_deepsurv(X_train_quad, T_train_quad, E_train_quad, n_trials=n_trials)
params_nonlinear_quad = best_nonlinear_quad.copy()


# whas
scaler_met = StandardScaler()
X_train_met_scaled = scaler_met.fit_transform(X_train_met)
best_met_params = optimise_deepsurv(X_train_met_scaled, T_train_met, E_train_met, n_trials=n_trials)
params_whas = best_met_params.copy()

# rossi
scaler_sup = StandardScaler()
X_train_sup_scaled = scaler_sup.fit_transform(X_train_sup)
best_sup_params = optimise_deepsurv(X_train_sup_scaled, T_train_sup, E_train_sup, n_trials=n_trials)
params_rossi = best_sup_params.copy()

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
# Print all optimal hyperparameters
print(f"params_linear = {params_linear}")
# print(f"params_nonlinear = {params_nonlinear}")
print(f"params_nonlinear_quad = {params_nonlinear_quad}")
print(f"params_whas = {params_whas}")
print(f"params_rossi = {params_rossi}")

In [ ]:
# params_linear = {'nodes': 96, 'layers': 3, 'dropout': 0.3926222949626741, 'lr': 0.0016079934171997832, 'l2': 0.025830820416880916, 'batch_size': 128, 'activation': 'ReLU', 'batch_norm': True}
# params_nonlinear_quad = {'nodes': 64, 'layers': 1, 'dropout': 0.2915174548967113, 'lr': 0.0010117585780204216, 'l2': 0.00010678744277989635, 'batch_size': 32, 'activation': 'ReLU', 'batch_norm': False}
# params_whas = {'nodes': 112, 'layers': 3, 'dropout': 0.3216732651595009, 'lr': 0.001411363988125655, 'l2': 0.0933122053491405, 'batch_size': 32, 'activation': 'SELU', 'batch_norm': True}
# params_rossi = {'nodes': 96, 'layers': 1, 'dropout': 0.4631103019151296, 'lr': 0.00015068056698091476, 'l2': 0.0022039075874811993, 'batch_size': 128, 'activation': 'ReLU', 'batch_norm': False}

In [ ]:
def optimise_deephit(X_train, T_train, E_train, n_trials=15):
    X_train = X_train.astype("float32")
    T_train = T_train.astype("float32")
    E_train = E_train.astype("float32")

    def objective(trial):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # parameters to optimise
        n_nodes = trial.suggest_int("nodes", 32, 64, step=32)
        n_layers = trial.suggest_int("layers", 1, 3)
        dropout = trial.suggest_float("dropout", 0.2, 0.6)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [64, 128])

        alpha = trial.suggest_float("alpha", 0.0, 1.0)
        sigma = trial.suggest_float("sigma", 0.1, 1.0)
        num_durations = trial.suggest_categorical("num_durations", [10, 20])

        # deephit
        labtrans = LabTransDiscreteTime(num_durations)
        y_train_discrete = labtrans.fit_transform(T_train, E_train)

        X_tr, X_val, y_tr_time, y_val_time, y_tr_event, y_val_event, T_tr_orig, T_val_orig, E_tr_orig, E_val_orig = train_test_split(
            X_train, y_train_discrete[0], y_train_discrete[1], T_train, E_train,
            test_size=0.2, random_state=42, stratify=y_train_discrete[1]
        )

        y_tr = (y_tr_time, y_tr_event)
        y_val = (y_val_time, y_val_event)

        # build the network
        in_features = X_tr.shape[1]
        out_features = labtrans.out_features
        num_nodes_list = [n_nodes] * n_layers

        net = tt.practical.MLPVanilla(
            in_features, num_nodes_list, out_features,
            batch_norm=True, dropout=dropout,
            activation=nn.ReLU, output_bias=False
        )

        optim = tt.optim.Adam(lr=lr)
        device = "cuda" if torch.cuda.is_available() else "cpu"

        model = DeepHitSingle(net, optim, alpha=alpha, sigma=sigma, duration_index=labtrans.cuts, device=device)
        callbacks = [tt.callbacks.EarlyStopping(patience=10)] # Lowered patience for speed/memory

        # train the model
        model.fit(
            X_tr, y_tr,
            batch_size=batch_size, epochs=200, # Lowered max epochs
            callbacks=callbacks, verbose=False,
            val_data=(X_val, y_val)
        )

        # compute risk
        surv_df = model.predict_surv_df(X_val)
        expected_survival_time = surv_df.sum().values
        risk = -expected_survival_time

        c_index = concordance_index(T_val_orig, risk, E_val_orig)

        # memory cleanup
        del model, net, optim, surv_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return c_index

    # Run the study
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    return study.best_params

In [ ]:
# Lowered n_trials for DeepHit to prevent Out-Of-Memory kernel crashes
n_trials_dh = 20

# Sim Linear
best_linear_dh = optimise_deephit(X_train_lin, T_train_lin, E_train_lin, n_trials=n_trials_dh)
params_linear_dh = best_linear_dh.copy()

# Sim Nonlinear Quadratic
best_nonlinear_quad_dh = optimise_deephit(X_train_quad, T_train_quad, E_train_quad, n_trials=n_trials_dh)
params_nonlinear_quad_dh = best_nonlinear_quad_dh.copy()


# whas
scaler_met = StandardScaler()
X_train_met_scaled = scaler_met.fit_transform(X_train_met)
best_met_params_dh = optimise_deephit(X_train_met_scaled, T_train_met, E_train_met, n_trials=n_trials_dh)
params_whas_dh = best_met_params_dh.copy()

# rossi
scaler_sup = StandardScaler()
X_train_sup_scaled = scaler_sup.fit_transform(X_train_sup)
best_sup_params_dh = optimise_deephit(X_train_sup_scaled, T_train_sup, E_train_sup, n_trials=n_trials_dh)
params_rossi_dh = best_sup_params_dh.copy()

In [ ]:
# Print all optimal hyperparameters
print(f"params_linear_dh = {params_linear_dh}")
# print(f"params_nonlinear = {params_nonlinear}")
print(f"params_nonlinear_quad_dh = {params_nonlinear_quad_dh}")
print(f"params_whas_dh = {params_whas_dh}")
print(f"params_rossi_dh = {params_rossi_dh}")

In [ ]:
# params_linear_dh = {'nodes': 32, 'layers': 2, 'dropout': 0.22937981467625856, 'lr': 0.00033416036208661743, 'batch_size': 128, 'alpha': 0.24531306225918015, 'sigma': 0.9427460024939718, 'num_durations': 10}
# params_nonlinear_quad_dh = {'nodes': 32, 'layers': 3, 'dropout': 0.5953575872008865, 'lr': 0.00011708866519224654, 'batch_size': 64, 'alpha': 0.44126113357661456, 'sigma': 0.7942692908443046, 'num_durations': 10}
# params_whas_dh = {'nodes': 32, 'layers': 3, 'dropout': 0.5541766066031479, 'lr': 0.00019026569115133615, 'batch_size': 128, 'alpha': 0.5977397895687814, 'sigma': 0.30835546690412413, 'num_durations': 20}
# params_rossi_dh = {'nodes': 64, 'layers': 1, 'dropout': 0.5920911102752608, 'lr': 0.0009301107552843405, 'batch_size': 64, 'alpha': 0.5042890877664027, 'sigma': 0.30436895887926285, 'num_durations': 20}

In [ ]:
def evaluate_dataset(X_train, T_train, E_train,
                     X_test, T_test, E_test,
                     dataset_name, params_ds, params_dh,
                     n_bootstraps=1000):

    # Scale features strictly on the training data to prevent leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # format targets for C-index
    T_test = np.array(T_test)
    E_test = np.array(E_test)

    # format targets for IBS
    y_train_sksurv = np.array([(bool(e), t) for e, t in zip(E_train, T_train)], dtype=[('e', bool), ('t', float)])
    y_test_sksurv = np.array([(bool(e), t) for e, t in zip(E_test, T_test)], dtype=[('e', bool), ('t', float)])

    # define the time grid for IBS integration (5th to 95th percentile of test events)
    mask = E_test == 1
    event_times = T_test[mask]
    times = np.linspace(np.percentile(event_times, 5), np.percentile(event_times, 95), 100)

    # train models and get predictions
    risk_cph, surv_cph = cph(X_train_scaled, T_train, E_train, X_test_scaled, times=times)
    risk_lasso, surv_lasso = cox_lasso(X_train_scaled, T_train, E_train, X_test_scaled, times=times)
    risk_ds, surv_ds = deepsurv(X_train_scaled, T_train, E_train, X_test_scaled, params_ds, times=times)
    risk_dh, surv_dh = deephit(X_train_scaled, T_train, E_train, X_test_scaled, params_dh, times=times)
    risk_rsf, surv_rsf = rsf(X_train_scaled, T_train, E_train, X_test_scaled, times=times)

    # Initialise lists for C-index
    cidx_cph, cidx_lasso, cidx_ds, cidx_rsf, cidx_dh = [], [], [], [], []
    # Initialise lists for IBS
    ibs_cph, ibs_lasso, ibs_ds, ibs_rsf, ibs_dh = [], [], [], [], []

    # Bootstrap loop
    for i in tqdm(range(n_bootstraps), desc=f"Bootstrapping {dataset_name}", leave=False):
        # Resample the test set indices with replacement
        indices = resample(np.arange(len(X_test_scaled)), replace=True, random_state=i)

        T_boot = T_test[indices]
        E_boot = E_test[indices]
        y_test_boot = y_test_sksurv[indices]

        if np.sum(E_boot) == 0:
            continue

        # concordance index
        cidx_cph.append(concordance_index(T_boot, -risk_cph[indices], E_boot))
        cidx_lasso.append(concordance_index(T_boot, -risk_lasso[indices], E_boot))
        cidx_ds.append(concordance_index(T_boot, -risk_ds[indices], E_boot))
        cidx_dh.append(concordance_index(T_boot, -risk_dh[indices], E_boot))
        cidx_rsf.append(concordance_index(T_boot, -risk_rsf[indices], E_boot))

        time_mask = (times >= T_boot.min()) & (times < T_boot.max())
        times_boot = times[time_mask]

        # Skip this bootstrap sample for IBS if it restricted the times too heavily (needs >= 2 points to integrate)
        if len(times_boot) < 2:
            continue

        # integrated brier score
        ibs_cph.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_cph[indices][:, time_mask], times_boot))
        ibs_lasso.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_lasso[indices][:, time_mask], times_boot))
        ibs_ds.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_ds[indices][:, time_mask], times_boot))
        ibs_dh.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_dh[indices][:, time_mask], times_boot))
        ibs_rsf.append(integrated_brier_score(y_train_sksurv, y_test_boot, surv_rsf[indices][:, time_mask], times_boot))

    # Helper function to extract Mean and 95% CI
    def get_metrics(boot_list):
        if not boot_list:
            return np.nan, (np.nan, np.nan)
        return round(np.mean(boot_list), 3), (round(np.percentile(boot_list, 2.5), 3), round(np.percentile(boot_list, 97.5), 3))

    return {
        "Dataset": dataset_name,
        # C-Index Results
        "C-Index CPH (Mean)": get_metrics(cidx_cph)[0], "C-Index CPH (95% CI)": get_metrics(cidx_cph)[1],
        "C-Index Cox Lasso (Mean)": get_metrics(cidx_lasso)[0], "C-Index Cox Lasso (95% CI)": get_metrics(cidx_lasso)[1],
        "C-Index DeepSurv (Mean)": get_metrics(cidx_ds)[0], "C-Index DeepSurv (95% CI)": get_metrics(cidx_ds)[1],
        "C-Index DeepHit (Mean)": get_metrics(cidx_dh)[0], "C-Index DeepHit (95% CI)": get_metrics(cidx_dh)[1],
        "C-Index RSF (Mean)": get_metrics(cidx_rsf)[0], "C-Index RSF (95% CI)": get_metrics(cidx_rsf)[1],

        # IBS Results
        "IBS CPH (Mean)": get_metrics(ibs_cph)[0], "IBS CPH (95% CI)": get_metrics(ibs_cph)[1],
        "IBS Cox Lasso (Mean)": get_metrics(ibs_lasso)[0], "IBS Cox Lasso (95% CI)": get_metrics(ibs_lasso)[1],
        "IBS DeepSurv (Mean)": get_metrics(ibs_ds)[0], "IBS DeepSurv (95% CI)": get_metrics(ibs_ds)[1],
        "IBS DeepHit (Mean)": get_metrics(ibs_dh)[0], "IBS DeepHit (95% CI)": get_metrics(ibs_dh)[1],
        "IBS RSF (Mean)": get_metrics(ibs_rsf)[0], "IBS RSF (95% CI)": get_metrics(ibs_rsf)[1],
    }

In [ ]:
n_bootstraps = 1000
final_results = []

# Sim Linear
final_results.append(evaluate_dataset(
    X_train_lin, T_train_lin, E_train_lin,
    X_test_lin, T_test_lin, E_test_lin,
    "Sim Linear", params_linear, params_linear_dh, n_bootstraps
))

# Sim Nonlinear Quadratic
final_results.append(evaluate_dataset(
    X_train_quad, T_train_quad, E_train_quad,
    X_test_quad, T_test_quad, E_test_quad,
    "Sim Nonlinear Quadratic", params_nonlinear_quad, params_nonlinear_quad_dh, n_bootstraps
))

# whas
final_results.append(evaluate_dataset(
    X_train_met, T_train_met, E_train_met,
    X_test_met, T_test_met, E_test_met,
    "whas", params_whas, params_whas_dh, n_bootstraps
))

# rossi
final_results.append(evaluate_dataset(
    X_train_sup, T_train_sup, E_train_sup,
    X_test_sup, T_test_sup, E_test_sup,
    "rossi", params_rossi, params_rossi_dh, n_bootstraps
))

# Convert the massive list of dictionaries into a single DataFrame
df_all = pd.DataFrame(final_results)

# --- NEW: Split the dataframe into two readable tables ---

# 1. Isolate the C-Index columns
df_cindex = df_all[['Dataset']].join(df_all.filter(like='C-Index'))
print("=========================================")
print("       CONCORDANCE INDEX RESULTS         ")
print("=========================================")
display(df_cindex) # use display() in Colab for a pretty HTML table
print("\n")

# 2. Isolate the IBS columns
df_ibs = df_all[['Dataset']].join(df_all.filter(like='IBS'))
print("=========================================")
print("    INTEGRATED BRIER SCORE RESULTS       ")
print("=========================================")
display(df_ibs)

Bootstrapping Sim Linear:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping Sim Nonlinear Quadratic:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping whas:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping TCGA:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping rossi:   0%|          | 0/1000 [00:00<?, ?it/s]

       CONCORDANCE INDEX RESULTS         


,Dataset,C-Index CPH (Mean),C-Index CPH (95% CI),C-Index Cox Lasso (Mean),C-Index Cox Lasso (95% CI),C-Index DeepSurv (Mean),C-Index DeepSurv (95% CI),C-Index DeepHit (Mean),C-Index DeepHit (95% CI),C-Index RSF (Mean),C-Index RSF (95% CI)
0,Sim Linear,0.786,"(0.77, 0.802)",0.786,"(0.771, 0.803)",0.785,"(0.769, 0.801)",0.781,"(0.765, 0.797)",0.779,"(0.764, 0.796)"
1,Sim Nonlinear Quadratic,0.509,"(0.487, 0.532)",0.509,"(0.486, 0.532)",0.772,"(0.754, 0.789)",0.760,"(0.741, 0.777)",0.768,"(0.749, 0.785)"
2,whas,0.836,"(0.768, 0.892)",0.838,"(0.772, 0.894)",0.817,"(0.746, 0.878)",0.422,"(0.33, 0.521)",0.809,"(0.742, 0.871)"
3,TCGA,0.606,"(0.526, 0.679)",0.607,"(0.529, 0.681)",0.606,"(0.532, 0.68)",0.631,"(0.56, 0.707)",0.630,"(0.555, 0.699)"
4,rossi,0.613,"(0.477, 0.733)",0.550,"(0.419, 0.681)",0.614,"(0.484, 0.73)",0.657,"(0.533, 0.777)",0.618,"(0.487, 0.725)"




    INTEGRATED BRIER SCORE RESULTS       


,Dataset,IBS CPH (Mean),IBS CPH (95% CI),IBS Cox Lasso (Mean),IBS Cox Lasso (95% CI),IBS DeepSurv (Mean),IBS DeepSurv (95% CI),IBS DeepHit (Mean),IBS DeepHit (95% CI),IBS RSF (Mean),IBS RSF (95% CI)
0,Sim Linear,0.132,"(0.123, 0.142)",0.132,"(0.123, 0.142)",0.132,"(0.123, 0.142)",0.433,"(0.412, 0.453)",0.135,"(0.126, 0.145)"
1,Sim Nonlinear Quadratic,0.216,"(0.206, 0.225)",0.216,"(0.206, 0.225)",0.138,"(0.129, 0.147)",0.273,"(0.261, 0.286)",0.144,"(0.135, 0.153)"
2,whas,0.138,"(0.104, 0.175)",0.139,"(0.105, 0.175)",0.148,"(0.114, 0.185)",0.234,"(0.185, 0.284)",0.159,"(0.128, 0.191)"
3,TCGA,0.222,"(0.182, 0.263)",0.221,"(0.181, 0.261)",0.211,"(0.173, 0.251)",0.203,"(0.169, 0.236)",0.201,"(0.172, 0.229)"
4,rossi,0.098,"(0.061, 0.14)",0.101,"(0.064, 0.145)",0.099,"(0.062, 0.142)",0.097,"(0.065, 0.136)",0.099,"(0.062, 0.141)"
